In [47]:
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import BertTokenizer
import ollama

ds = load_dataset("cardiffnlp/tweet_topic_single")
train = ds["train_all"]

label_names = train.features["label"].names

print("Labels    :", label_names)
print("Train size:", len(train))
print("Sample 0  :", train[0])

Labels    : ['arts_&_culture', 'business_&_entrepreneurs', 'pop_culture', 'daily_life', 'sports_&_gaming', 'science_&_technology']
Train size: 4374
Sample 0  : {'text': 'The {@Clinton LumberKings@} beat the {@Cedar Rapids Kernels@} 4-0 in Game 1 of the Western Division finals. Evan Edwards hit a 2-run HR. WP Josh Roberson: 5 IP, 3 H, 0 R, 0 BB, 10 K #MWLplayoffs #MWLscoreboard', 'date': '2019-09-08', 'label': 4, 'label_name': 'sports_&_gaming', 'id': '1170516324419866624'}



## TF-IDF Demonstration (Logistic Regression)

TF-IDF or Term Frequency - Inverse Document Frequency is a metric used to convert a piece of text into a numeric vector to be used by machine learning models as an embedding. TF-IDF is calculated in two parts, the TF score and IDF score. TF refers to how often a given term or word appears in a document. Take the number of times a term appears in a document and divide that by the number of terms in that document. The other component is the IDF score which is calculated by taking the log of the number of documents in the corpus and divide that by the number of documents containing a specific term. This tells you how important a word is across the corpus. You can calculate the TF-IDF score by multiplying these two scores. A document is turned into a feature vector by calculating the TF-IDF score of every word in a document and storing them as a vector.

In [56]:
n = 3
while True:
    samples = train.select(range(n))
    texts   = samples["text"]
    labels  = [label_names[s["label"]] for s in samples]

    vec   = TfidfVectorizer()
    X     = vec.fit_transform(texts)
    terms = vec.get_feature_names_out()

    if len(terms) >= 250:
        break
    n += 1


print(f"Corpus used for fitting  : {n} tweets")
print(f"Vocabulary (unique terms): {len(terms)}")

Corpus used for fitting  : 14 tweets
Vocabulary (unique terms): 253


In [57]:
import numpy as np

raw_idf   = vec.idf_
idf_min, idf_max = raw_idf.min(), raw_idf.max()
norm_idf  = (raw_idf - idf_min) / (idf_max - idf_min)

idf_scores = list(zip(terms, norm_idf))
idf_scores.sort(key=lambda x: -x[1])

print(f"Corpus: {n} tweets   |   Vocabulary: {len(terms)} unique terms")
print(f"{'TERM':<30} {'NORM IDF':>10}")
print(f"{'─'*30} {'─'*10}")
for term, score in idf_scores:
    print(f"{term:<30} {score:>10.6f}")

Corpus: 14 tweets   |   Vocabulary: 253 unique terms
TERM                             NORM IDF
────────────────────────────── ──────────
000                              1.000000
10                               1.000000
12                               1.000000
1s                               1.000000
2023                             1.000000
23                               1.000000
2s                               1.000000
31                               1.000000
50                               1.000000
60                               1.000000
65not                            1.000000
ago                              1.000000
aims                             1.000000
allows                           1.000000
announce                         1.000000
anyone                           1.000000
are                              1.000000
arena                            1.000000
arguably                         1.000000
arkansas                         1.000000
as                     

The TF-IDF scores were normalized across the corpus to go from 0 to 1, with 0 being the most common and 1 being rare descriptive words. So specific sets of numbers and things like hashtags that only appear 1 have the highest scores, and common words like 'the', 'to', or 'username' that appear quite often in tweets have low scores and aren't as important to a machine learning model.

## BERT Tokenization

In [58]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

TARGET_CATEGORIES = {
    "sports_&_gaming":      5,
    "science_&_technology": 5,
}
category_counts = {c: 0 for c in TARGET_CATEGORIES}
selected = []
for item in train:
    lbl_name = label_names[item["label"]]
    if lbl_name in category_counts and category_counts[lbl_name] < TARGET_CATEGORIES[lbl_name]:
        selected.append(item)
        category_counts[lbl_name] += 1
    if all(category_counts[c] >= TARGET_CATEGORIES[c] for c in TARGET_CATEGORIES):
        break

results = []
for item in selected:
    text     = item["text"]
    tokens   = tokenizer.tokenize(text)
    subwords = [t for t in tokens if t.startswith("##")]
    results.append({"text": text, "label": label_names[item["label"]], "tokens": tokens, "subwords": subwords})

has_split = any(len(r["subwords"]) > 0 for r in results)
if not has_split:
    hardcoded   = "The unhappiest, most unreasonable superpowered #antiheroism storyline"
    tokens_hc   = tokenizer.tokenize(hardcoded)
    subwords_hc = [t for t in tokens_hc if t.startswith("##")]
    results.append({"text": hardcoded, "label": "[hardcoded]", "tokens": tokens_hc, "subwords": subwords_hc})
    print("NOTE: No subword splits found — appended hardcoded example.\n")

for i, r in enumerate(results):
    flagged = [f"{t}*" if t.startswith("##") else t for t in r["tokens"]]

    print(f"Tweet {i+1}  (label: {r['label']})")
    print(f"Original : {r['text'][:120]}{'...' if len(r['text']) > 120 else ''}")
    print(f"Tokens   : {flagged}")
    print()

Tweet 1  (label: sports_&_gaming)
Original : The {@Clinton LumberKings@} beat the {@Cedar Rapids Kernels@} 4-0 in Game 1 of the Western Division finals. Evan Edwards...
Tokens   : ['the', '{', '@', 'clinton', 'lumber', '##king*', '##s*', '@', '}', 'beat', 'the', '{', '@', 'cedar', 'rapids', 'kernel', '##s*', '@', '}', '4', '-', '0', 'in', 'game', '1', 'of', 'the', 'western', 'division', 'finals', '.', 'evan', 'edwards', 'hit', 'a', '2', '-', 'run', 'hr', '.', 'w', '##p*', 'josh', 'robe', '##rson*', ':', '5', 'ip', ',', '3', 'h', ',', '0', 'r', ',', '0', 'bb', ',', '10', 'k', '#', 'mw', '##lp*', '##lay*', '##offs*', '#', 'mw', '##ls*', '##core*', '##board*']

Tweet 2  (label: sports_&_gaming)
Original : I would rather hear Eli Gold announce this Auburn game than these dumbasses. {@ESPN@}
Tokens   : ['i', 'would', 'rather', 'hear', 'eli', 'gold', 'announce', 'this', 'auburn', 'game', 'than', 'these', 'dumb', '##asse*', '##s*', '.', '{', '@', 'espn', '@', '}']

Tweet 3  (label: sports_&_g

The tokenizer handled most responses well, and it could split words properly as seen in tweet 1 and tweet 3. But it did struggle with certain aspects of the tweets. Anytime a word got replaced with something like {{USERNAME}} or {{@ESPN@}}, the tokenizer would break each outer character into their own tokens.
'{', '@', 'espn', '@', '}'
I'm not sure how this will affect the fine tuning of the model or the accuracy of the model.

## Llama3 Zero-Shot Prompt Classification
Need Ollama installed as a prerequisite. I used llama3/latest for this example, but other models are suitable. You just need to replace the value in OLLAMA_MODEL

In [31]:
def build_prompt(tweet: str, label_names: list) -> str:
    """Build a zero-shot classification prompt.
    Label names are injected at runtime — never hardcoded.
    """
    labels_str = "\n".join(f"- {l}" for l in label_names)
    return (
        "You are a tweet classifier. Classify the following tweet into exactly "
        f"one of these categories:\n\n{labels_str}\n\n"
        "Respond with ONLY the category label. No explanation, no punctuation.\n\n"
        f"Tweet: {tweet}\n\nCategory:"
    )

sample0_text   = train[0]["text"]
sample0_prompt = build_prompt(sample0_text, label_names)

print("FULL RAW PROMPT")
print("─" * 70)
print(sample0_prompt)

FULL RAW PROMPT
──────────────────────────────────────────────────────────────────────
You are a tweet classifier. Classify the following tweet into exactly one of these categories:

- arts_&_culture
- business_&_entrepreneurs
- pop_culture
- daily_life
- sports_&_gaming
- science_&_technology

Respond with ONLY the category label. No explanation, no punctuation.

Tweet: The {@Clinton LumberKings@} beat the {@Cedar Rapids Kernels@} 4-0 in Game 1 of the Western Division finals. Evan Edwards hit a 2-run HR. WP Josh Roberson: 5 IP, 3 H, 0 R, 0 BB, 10 K #MWLplayoffs #MWLscoreboard

Category:


In [ ]:
OLLAMA_MODEL  = "llama3.1:latest"
TARGET_LABELS = 3

label_to_sample_llm = {}
for item in train:
    lbl = item["label"]
    if lbl not in label_to_sample_llm:
        label_to_sample_llm[lbl] = item
    if len(label_to_sample_llm) == TARGET_LABELS:
        break

llm_samples = [label_to_sample_llm[i] for i in sorted(label_to_sample_llm)]

for item in llm_samples:
    tweet    = item["text"]
    true_lbl = label_names[item["label"]]
    prompt   = build_prompt(tweet, label_names)

    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[{"role": "user", "content": prompt}]
    )
    raw_out = response.message.content.strip()

    print(f"True: {true_lbl}")
    print(f"Tweet: {tweet[:120]}{'...' if len(tweet) > 120 else ''}")
    print(f"Llama3 out: {raw_out}")
    print()

True: pop_culture
Tweet: Y’all leave {@50cent@} alone about that theme song. He done change the song back to the Joe version. ‍♀️ #power
Llama3 out: pop_culture

True: daily_life
Tweet: 1st of all, was good to see you {@stellacreasy@} in Walthamstow not Bournemouth, as local resident, I tried to contact y...
Llama3 out: daily_life

True: sports_&_gaming
Tweet: The {@Clinton LumberKings@} beat the {@Cedar Rapids Kernels@} 4-0 in Game 1 of the Western Division finals. Evan Edwards...
Llama3 out: sports_&_gaming

